In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

In [20]:
from typing import TypedDict,Literal
from dotenv import load_dotenv

In [14]:
import os
load_dotenv(override=True)

True

In [15]:
def get_groq_llm():
    return ChatOpenAI(
        model= "openai/gpt-oss-120b",
        base_url= "https://api.groq.com/openai/v1",
        api_key= os.getenv("GROQ_API_KEY"),
        max_tokens= 1000
    )

llm= get_groq_llm()

In [16]:
from pydantic import BaseModel, Field

In [21]:
class SentimentSchema(BaseModel):
    sentiment:Literal["positive","negative"]=Field(description="sentiment of the review")

In [22]:
class DiagnosisSchema(BaseModel):
    issue_type:Literal["UX/UI","Performance","Bug","Other"]=Field(description="The category of the issue mentioned in the review")
    tone:Literal["angry","frustrated","disappointed","calm"]=Field(description="The emotional tone expressed by the user review")
    urgency:Literal["low","medium","high"]=Field(description="How urgent or critical the issues appears to be")

In [23]:
structured_model_1=llm.with_structured_output(SentimentSchema)
structured_model_2=llm.with_structured_output(DiagnosisSchema)

In [24]:
review="It was really bad product"
structured_model_1.invoke(review).sentiment

'negative'

In [25]:
structured_model_2.invoke(review)

DiagnosisSchema(issue_type='Other', tone='disappointed', urgency='medium')

In [26]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict 
    response:str

In [27]:
def find_sentiment(state:ReviewState):
    prompt=f'For the following review, find out the sentiment {state["review"]}'
    sentiment=structured_model_1.invoke(prompt).sentiment
    return {"sentiment":sentiment}

In [28]:
## helper function
def check_sentiment(state:ReviewState):
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [ ]:
def positive_response(state:ReviewState):
    prompt=f"Write a warm thank you message in your response based on the review of the user: {state['review']}"
    response=llm.invoke(prompt).content
    return {'response':response} 

In [30]:
def run_diagnosis(state:ReviewState):
    prompt=f"""
    Diagnose this negative review:\n 
    {state['review']}
    """
    raw_response=structured_model_2.invoke(prompt)
    return {'diagnosis':raw_response.model_dump()} 

In [32]:
def negative_response(state:ReviewState):
    diagnosis=state['diagnosis']
    prompt=f"""
    You are a support assistant.
    The user has a {diagnosis['issue_type']}, sounded {diagnosis['tone']}, and marked urgency as 
    {diagnosis['urgency']}. Write an empathetic, helpful response message.
    """
    response=llm.invoke(prompt).content
    return {"response":response}

In [33]:
graph=StateGraph(ReviewState)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

In [ ]:
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)